# Join network logons to the processes they started (Spark SQL)

- Load Windows logon events and process creation events
- Join them on the logon session identifier with Spark SQL
- See which remote logons led to process execution, and from where

---
<sub>TechDetechtives analytics workbench. This notebook is a modified work derived from the HELK notebook `demos/security_sysmon_sql_join.ipynb` (https://github.com/Cyb3rWard0g/HELK), Copyright Roberto Rodriguez (@Cyb3rWard0g), licensed GPL-3.0. Modified by TechDetechtives, 2026: reads from the platform through a read-only client instead of the Spark Elasticsearch connector, uses ECS field names, and runs Spark in local mode. Licensed GPL-3.0; see `analytics/LICENSE`.</sub>

## Fields used

| Meaning | Logon events (Security 4624) | Process events (Sysmon 1) |
| --- | --- | --- |
| Logon session | `winlog.event_data.TargetLogonId` | `winlog.event_data.LogonId` |
| Logon type (3 = network) | `winlog.event_data.LogonType` | |
| Origin address | `source.ip` | |

The join needs Sysmon process events, because they carry the logon session identifier.

In [ ]:
import td_hunt as td

spark = td.spark_session("TechDetechtives logon join")
spark

## Load logon events and register a view

In [ ]:
logon_df = td.events(
    "logon",
    since="24h",
    fields=["host.name", "user.name", "source.ip", "event.code",
            "winlog.event_data.LogonType", "winlog.event_data.TargetLogonId"],
)
td.to_spark(logon_df, view="logon", spark=spark).printSchema()

## Load process creation events and register a view

In [ ]:
process_df = td.events(
    "process",
    since="24h",
    fields=["host.name", "user.name", "process.name", "process.command_line",
            "process.parent.name", "process.entity_id", "process.parent.entity_id",
            "winlog.event_data.LogonId"],
)
td.to_spark(process_df, view="process", spark=spark).printSchema()

## Network logons with a known origin

In [ ]:
network_logons = spark.sql("""
    SELECT timestamp, host_name, user_name, source_ip, winlog_event_data_TargetLogonId AS logon_id
    FROM logon
    WHERE event_code = '4624'
      AND winlog_event_data_LogonType = '3'
      AND source_ip IS NOT NULL
""")
network_logons.createOrReplaceTempView("network_logons")
network_logons.show(truncate=False)

## Join: which of those logons started processes?

In [ ]:
joined = spark.sql("""
    SELECT l.timestamp AS logon_time,
           l.host_name,
           l.source_ip,
           l.user_name,
           p.timestamp AS process_time,
           p.process_parent_name,
           p.process_name,
           p.process_command_line
    FROM network_logons l
    INNER JOIN process p
            ON p.winlog_event_data_LogonId = l.logon_id
           AND p.host_name = l.host_name
    ORDER BY p.timestamp
""")
joined.select("source_ip", "host_name", "user_name", "process_parent_name", "process_name").show(truncate=False)

## Summary per origin address

In [ ]:
joined.createOrReplaceTempView("joined")
spark.sql("""
    SELECT source_ip, host_name, user_name, COUNT(*) AS processes_started
    FROM joined
    GROUP BY source_ip, host_name, user_name
    ORDER BY processes_started DESC
""").show(truncate=False)

## Full command lines for review

In [ ]:
joined.select("process_time", "process_name", "process_command_line").toPandas()